In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import constantes as const
from utils.preprocess import DataPreprocessor
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


In [2]:
from utils.data import DataManager

data_manager = DataManager()
data_manager.preparar_conjuntos()

X_train, y_train, weights_train = data_manager.get_train_data()
X, y = data_manager.get_X_y()
test = data_manager.get_test()
weights = data_manager.get_weights()
X_test, y_test, weights_test = data_manager.get_test_data()


In [3]:
data_preprocessor = DataPreprocessor()

X_train = data_preprocessor.fit_transform(X_train)
X = data_preprocessor.transform(X)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import make_scorer
from sklearn.ensemble import ExtraTreesClassifier
import numpy as np
import sklearn

import importlib
import utils.comet_experiments as comet_exp
import utils.HiggsBosonCompetition_AMSMetric_rev1 as hb_metrics

importlib.reload(hb_metrics)
importlib.reload(comet_exp)

sklearn.set_config(enable_metadata_routing=True)

ams_custom_score = make_scorer(hb_metrics.ams_scorer, greater_is_better=True, needs_proba=False).set_score_request(sample_weight=True)

X_train, X_val, y_train, y_val, weights_train, weights_val = train_test_split(
    X_train, y_train, weights_train, test_size=0.1, random_state=42
)

model = ExtraTreesClassifier(
    n_estimators=1000,
    random_state=42,
    n_jobs=-1
).set_fit_request(sample_weight=True)

param_dist = {
    'max_depth': np.arange(3, 15),
    'min_samples_split': np.arange(2, 20, 2),
    'min_samples_leaf': np.arange(1, 10, 1),
    'max_features': ['auto', 'sqrt', 'log2']
}
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=50,
    scoring=ams_custom_score,
    cv=3,
    verbose=3,
    random_state=42
)

random_search.fit(X_train, y_train, sample_weight=weights_train)
print("Mejores parámetros:", random_search.best_params_)
print("Mejor puntaje:", random_search.best_score_)

sklearn.set_config(enable_metadata_routing=False)
